### 환경설정
- chroma(벡터Db) : 로컬 인메모리 모드지원, 서버불필요
- Neo4j python driver : Neo4j 서버통신
- sentence-transformers : 텍스트 임베딩
- sckit-learn : PCA 데이터 분석 

In [ ]:
!pip install chromadb sentence-transformers neo4j matplotlib networkx scikit-learn -q

In [2]:
# 설치확인
import chromadb
import neo4j
import sentence_transformers
print(chromadb.__version__, neo4j.__version__, sentence_transformers.__version__)


1.5.9 6.2.0 5.5.1


### 벡터DB
- 고차원벡터(Embedding)를 저장, 유사도검색을 수행하는데 특화
- RAG 핵심 인프라 : LLM의 외부 지식검색에 필수적
- 메타데이터 필터링 : 벡터검색 + 조건 필터링 동시 지원

### 벡터DB 종류
- Chroma : 경량                : 학습(프로토타입)
- Pinecone : 관리형 ,고 가용성   : 대용량 프로젝트
- Weaviate : 멀티모달           : 복합검색
- Milvus : 분선처리 GPU         : 대용량 프로젝트
- FAISS : Meta 오픈소스, 초고속  : 연구용(벤치마크)

In [3]:
import chromadb
client = chromadb.Client()  # in-memory
# client = chromadb.PersistentClient(path='./outputs')
client.heartbeat()  # 

1779759801272309400

### collection  생성
- 벡터들의 논리적 그룹(RDBMS 테이블 개념)
- 거리함수 : cosine(기본)
client

In [ ]:
collection = client.get_or_create_collection(
    name = 'korean_foods',
    metadata={'hnsw:space':'cosine'}
)
print(f'컬렉션 이름 : {collection.name}') 
print(f'현재 문서 수 : {collection.count()}')

컬렉션 이름 : forean_foods
현재 문서 수 : 0


In [5]:
documents = [
    "김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다.",
    "불고기는 간장 양념에 재운 소고기를 구워 먹는 한국 전통 요리입니다.",
    "비빔밥은 밥 위에 다양한 나물과 고추장을 넣고 비벼 먹는 음식입니다.",
    "된장찌개는 된장을 풀어 두부, 감자, 호박 등을 넣고 끓인 찌개입니다.",
    "삼겹살은 돼지 뱃살을 구워 쌈 채소에 싸서 먹는 인기 있는 요리입니다.",
    "떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다.",
    "냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다.",
    "잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다.",
    "갈비탕은 소갈비를 오랫동안 끓여 만든 깊은 맛의 탕 요리입니다.",
    "순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다.",
]

metadatas = [
    {"category": "찌개", "main_ingredient": "돼지고기", "spicy": True},
    {"category": "구이", "main_ingredient": "소고기", "spicy": False},
    {"category": "밥",  "main_ingredient": "채소",   "spicy": True},
    {"category": "찌개", "main_ingredient": "된장",   "spicy": False},
    {"category": "구이", "main_ingredient": "돼지고기", "spicy": False},
    {"category": "분식", "main_ingredient": "떡",     "spicy": True},
    {"category": "면",  "main_ingredient": "메밀",   "spicy": False},
    {"category": "볶음", "main_ingredient": "당면",   "spicy": False},
    {"category": "탕",  "main_ingredient": "소고기", "spicy": False},
    {"category": "찌개", "main_ingredient": "순두부", "spicy": True},
]

ids = [f'food_{i:03d}' for i in range(len(documents))]
print(f'ids = {ids}')

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)
print(f'{collection.count()}개 문서 추가 완료')
for doc_id, doc in zip(ids,documents):
    print(f'    {doc_id} : {doc[:30]}...')

ids = ['food_000', 'food_001', 'food_002', 'food_003', 'food_004', 'food_005', 'food_006', 'food_007', 'food_008', 'food_009']


C:\Users\Playdata\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:09<00:00, 8.63MiB/s]


10개 문서 추가 완료
    food_000 : 김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다.
    food_001 : 불고기는 간장 양념에 재운 소고기를 구워 먹는 한국 전통 요리입니다.
    food_002 : 비빔밥은 밥 위에 다양한 나물과 고추장을 넣고 비벼 먹는 음식입니다.
    food_003 : 된장찌개는 된장을 풀어 두부, 감자, 호박 등을 넣고 끓인 찌개입니다.
    food_004 : 삼겹살은 돼지 뱃살을 구워 쌈 채소에 싸서 먹는 인기 있는 요리입니다.
    food_005 : 떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다.
    food_006 : 냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다.
    food_007 : 잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다.
    food_008 : 갈비탕은 소갈비를 오랫동안 끓여 만든 깊은 맛의 탕 요리입니다.
    food_009 : 순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다.


### 유사도 검색

In [8]:
query  = '매운 국물 요리가 먹고 싶어요'
results = collection.query(
    query_texts=[query],
    n_results=5  # top_5
)
for doc, meta,dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
    similarity = 1- dist # 코사인거리->유사도 변환    
    print(similarity,doc,meta,dist)

0.8424747586250305 떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다. {'category': '분식', 'main_ingredient': '떡', 'spicy': True} 0.15752524137496948
0.809877336025238 순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다. {'category': '찌개', 'spicy': True, 'main_ingredient': '순두부'} 0.19012266397476196
0.7872368097305298 김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다. {'spicy': True, 'category': '찌개', 'main_ingredient': '돼지고기'} 0.21276319026947021
0.7770899534225464 냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다. {'main_ingredient': '메밀', 'spicy': False, 'category': '면'} 0.2229100465774536
0.7485682368278503 잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다. {'category': '볶음', 'spicy': False, 'main_ingredient': '당면'} 0.25143176317214966


In [11]:
queries = [
    "고기를 구워서 먹는 음식",
    "시원한 여름 음식 추천해주세요",
    "명절에 먹는 전통 음식",
]

# 각 질문에 대해서 top-3 문장을 출력
for query in queries:
    results = collection.query(
        query_texts=[query],
        n_results=3  # top_5
    )
    print(f'\n질문 : {query}')
    for doc, meta,dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
        similarity = 1- dist # 코사인거리->유사도 변환            
        print(f'유사도({similarity}) 문서:{doc[:15]}...,메타:{meta}')


질문 : 고기를 구워서 먹는 음식
유사도(0.7897407412528992) 문서:불고기는 간장 양념에 재운 ...,메타:{'main_ingredient': '소고기', 'category': '구이', 'spicy': False}
유사도(0.7736005783081055) 문서:떡볶이는 떡과 어묵을 고추장...,메타:{'main_ingredient': '떡', 'category': '분식', 'spicy': True}
유사도(0.7421424388885498) 문서:비빔밥은 밥 위에 다양한 나...,메타:{'spicy': True, 'category': '밥', 'main_ingredient': '채소'}

질문 : 시원한 여름 음식 추천해주세요
유사도(0.7804622650146484) 문서:비빔밥은 밥 위에 다양한 나...,메타:{'main_ingredient': '채소', 'category': '밥', 'spicy': True}
유사도(0.7059831023216248) 문서:냉면은 메밀 면을 차가운 육...,메타:{'main_ingredient': '메밀', 'category': '면', 'spicy': False}
유사도(0.6690664887428284) 문서:김치찌개는 돼지고기와 김치를...,메타:{'category': '찌개', 'main_ingredient': '돼지고기', 'spicy': True}

질문 : 명절에 먹는 전통 음식
유사도(0.773314356803894) 문서:비빔밥은 밥 위에 다양한 나...,메타:{'main_ingredient': '채소', 'category': '밥', 'spicy': True}
유사도(0.7512445449829102) 문서:냉면은 메밀 면을 차가운 육...,메타:{'main_ingredient': '메밀', 'category': '면', 'spicy': False}
유사도(0.7327145338058472) 문서:떡볶이는 떡과 어묵을 고추장...,메타:{'main_ingredient': '떡', '

### 메타데이터 필터링
- 기본 벡터유사도 검색 + 조건필터
- 문맥을 통해 유사한 문장을 찾으면서 특정조건을 만족하는 결과

In [12]:
# 매운음식만 검색 : 필터를 매운음식중에 국물요리 
results = collection.query(
        query_texts=["따뜻한 국물 요리"],
        n_results=3,  # top_5
        where={'spicy':True}  # 매운 음식만
    )
for doc, meta,dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
        similarity = 1- dist # 코사인거리->유사도 변환            
        print(f'유사도({similarity}) 문서:{doc[:15]}...,메타:{meta}')

유사도(0.7249371409416199) 문서:김치찌개는 돼지고기와 김치를...,메타:{'main_ingredient': '돼지고기', 'spicy': True, 'category': '찌개'}
유사도(0.6877250671386719) 문서:순두부찌개는 부드러운 순두부...,메타:{'category': '찌개', 'main_ingredient': '순두부', 'spicy': True}
유사도(0.6661773324012756) 문서:떡볶이는 떡과 어묵을 고추장...,메타:{'category': '분식', 'spicy': True, 'main_ingredient': '떡'}


In [13]:
# 카테고리가 찌개인 음식 
# 카테고리가 구이 인 음식
# 소고기 또는 돼지고기를 사용하고 + 매운 음식이 아닌 것

In [17]:
results = collection.query(
        query_texts=["고기종류의 음식"],
        n_results=3,  # top_5
        where={
                "$and": [
                    { 'main_ingredient':{"$in":["소고기","돼지고기"]} },
                    { 'spicy':False }
                ]
            }
                
    )
for doc, meta,dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
        similarity = 1- dist # 코사인거리->유사도 변환            
        print(f'유사도({similarity}) 문서:{doc[:15]}...,메타:{meta}')

유사도(0.49335598945617676) 문서:불고기는 간장 양념에 재운 ...,메타:{'main_ingredient': '소고기', 'spicy': False, 'category': '구이'}
유사도(0.47859489917755127) 문서:갈비탕은 소갈비를 오랫동안 ...,메타:{'spicy': False, 'category': '탕', 'main_ingredient': '소고기'}
유사도(0.3948414921760559) 문서:삼겹살은 돼지 뱃살을 구워 ...,메타:{'category': '구이', 'main_ingredient': '돼지고기', 'spicy': False}


### 커스텀 임베딩 모델 사용
- Chroma 기본 임베딩 대신 Sentence-Transformer 다국어 모델을 사용해서 한국어 검색 성능을 향상

In [18]:
from chromadb.utils import embedding_functions
st_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# 커스텀 임베딩 모델용 컬렉션 생성
collection_custom =  client.get_or_create_collection(
    name = 'korean_foods_custom',
    embedding_function=st_ef,
    metadata={'hnsw:space':"cosine"}
)
# 문서추가
collection_custom.add(
    documents=documents,
    metadatas=metadatas,
    ids = [f'custom_{i:03d}' for i in range(len(documents))]
)
# 기본 vs 커스텀 임베딩
test_query = '뜨끈한 국물이 있는 겨울 음식'
r1 = collection.query(query_texts=[test_query],n_results=3)
r2 = collection_custom.query(query_texts=[test_query],n_results=3)

def showResult(results):
    for doc, meta,dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
            similarity = 1- dist # 코사인거리->유사도 변환            
            print(f'유사도({similarity}) 문서:{doc[:15]}...,메타:{meta}')   
showResult(r1)
print('\n')
showResult(r2)